In [23]:
import pandas as pd
import numpy as np

# โหลดข้อมูล
print("📊 โหลดข้อมูล...")
df = pd.read_csv('btc_training_data.csv')
print(f"✅ โหลด {len(df):,} rows")
print(f"📋 Columns: {df.columns.tolist()}")

📊 โหลดข้อมูล...
✅ โหลด 995,105 rows
📋 Columns: ['timestamp_ms', 'readable_time', 'price', 'quantity', 'side', 'is_maker']


In [24]:
# สร้าง timestamp_sec
df['timestamp_sec'] = (df['timestamp_ms'] // 1000).astype(int)

# สร้าง signed_qty (BUY = +, SELL = -)
df['signed_qty'] = df.apply(
    lambda row: row['quantity'] if row['side'] == 'BUY' else -row['quantity'], 
    axis=1
)

print(f"✅ สร้าง Column เรียบร้อย")
print(df.head())
print(f"\n📋 Columns: {df.columns.tolist()}")

✅ สร้าง Column เรียบร้อย
    timestamp_ms               readable_time     price  quantity  side  \
0  1768729385585  2026-01-18 09:43:05.585000  95238.62   0.00006  SELL   
1  1768729385932  2026-01-18 09:43:05.932000  95238.62   0.05775  SELL   
2  1768729387628  2026-01-18 09:43:07.628000  95238.63   0.00006   BUY   
3  1768729388109  2026-01-18 09:43:08.109000  95238.63   0.00021   BUY   
4  1768729389444  2026-01-18 09:43:09.444000  95238.62   0.00100  SELL   

   is_maker  timestamp_sec  signed_qty  
0      True     1768729385    -0.00006  
1      True     1768729385    -0.05775  
2     False     1768729387     0.00006  
3     False     1768729388     0.00021  
4      True     1768729389    -0.00100  

📋 Columns: ['timestamp_ms', 'readable_time', 'price', 'quantity', 'side', 'is_maker', 'timestamp_sec', 'signed_qty']


In [25]:
print("\n🔄 Aggregate เป็นรายวินาที...")

agg_df = df.groupby('timestamp_sec').agg({
    'quantity': 'sum',           # total_volume
    'signed_qty': 'sum',         # net_flow
    'price': ['count', 'last']   # trade_count, close_price
}).reset_index()

# Flatten column names
agg_df.columns = ['timestamp_sec', 'total_volume', 'net_flow', 'trade_count', 'close_price']

print(f"✅ Aggregated: {len(agg_df):,} rows (วินาที)")
print(agg_df.head(10))


🔄 Aggregate เป็นรายวินาที...
✅ Aggregated: 103,890 rows (วินาที)
   timestamp_sec  total_volume  net_flow  trade_count  close_price
0     1768729385       0.05781  -0.05781            2     95238.62
1     1768729387       0.00006   0.00006            1     95238.63
2     1768729388       0.00021   0.00021            1     95238.63
3     1768729389       0.00100  -0.00100            1     95238.62
4     1768729390       0.14303   0.14303            2     95238.63
5     1768729391       0.00041  -0.00041            2     95238.62
6     1768729392       0.00176  -0.00134            3     95238.62
7     1768729393       0.00212   0.00080            3     95238.62
8     1768729394       0.00012   0.00012            1     95238.63
9     1768729395       0.01913   0.00189            2     95238.63


In [27]:
HOLDING_TIME = 5

print(f"\n🏷️ สร้าง Label (ราคา {HOLDING_TIME} วินาทีข้างหน้า)...")

agg_df['future_price'] = agg_df['close_price'].shift(-HOLDING_TIME)
agg_df['price_change'] = agg_df['future_price'] - agg_df['close_price']
agg_df['label'] = (agg_df['price_change'] > 0).astype(int)

# ลบ rows ที่ไม่มี future price
agg_df = agg_df.dropna()

print(f"✅ Label distribution:")
print(f"   0 (ลง): {sum(agg_df['label']==0):,}")
print(f"   1 (ขึ้น): {sum(agg_df['label']==1):,}")


🏷️ สร้าง Label (ราคา 5 วินาทีข้างหน้า)...
✅ Label distribution:
   0 (ลง): 70,052
   1 (ขึ้น): 33,828


In [29]:
from sklearn.preprocessing import StandardScaler

feature_cols = ['total_volume', 'net_flow', 'trade_count']
X = agg_df[feature_cols].values
y = agg_df['label'].values

print(f"✅ Features: {feature_cols}")
print(f"✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"✅ Normalized!")

✅ Features: ['total_volume', 'net_flow', 'trade_count']
✅ X shape: (103880, 3)
✅ y shape: (103880,)
✅ Normalized!


In [30]:
SEQUENCE_LENGTH = 10  # ดูย้อนหลัง 10 วินาที

def create_sequences(X, y, seq_length):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i:i+seq_length])
        ys.append(y[i+seq_length])
    return np.array(Xs), np.array(ys)

print(f"🔄 สร้าง Sequences (ย้อนหลัง {SEQUENCE_LENGTH} วินาที)...")
X_seq, y_seq = create_sequences(X_scaled, y, SEQUENCE_LENGTH)
print(f"✅ X_seq shape: {X_seq.shape}")
print(f"✅ y_seq shape: {y_seq.shape}")

🔄 สร้าง Sequences (ย้อนหลัง 10 วินาที)...
✅ X_seq shape: (103870, 10, 3)
✅ y_seq shape: (103870,)


In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

print(f"📊 Train: {len(X_train):,}")
print(f"📊 Test: {len(X_test):,}")

📊 Train: 83,096
📊 Test: 20,774


In [32]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

print("🧠 สร้าง LSTM Model...")

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, len(feature_cols))),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

model.summary()

🧠 สร้าง LSTM Model...


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 10, 64)         │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,369 (118.63 KB)

 Trainable params: 30,369 (118.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

print("🚀 เริ่ม Training...")

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

🚀 เริ่ม Training...
Epoch 1/50
